# Load ESM-2 # NEED TO REGEN ORIGINAL EMBDS

In [200]:
#gen ESM-2 embeddings for training sequences
import torch
import esm
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler
import torch.nn as nn

# load device GPU/CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.10.0


In [201]:
#load esm-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.to(DEVICE)
model.eval()

# prep model for inference
last_layer = len(model.layers)
batch_converter = alphabet.get_batch_converter()  # convert seqs to tokens

In [202]:
def tokenize_batch(sequences):
    # convert a batch of sequences to tokens (MUCH faster than one-at-a-time)
    batch = [(f"orf{i}", seq) for i, seq in enumerate(sequences)]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch)
    return batch_tokens.to(device=DEVICE, dtype=torch.long)

In [203]:
def extract_embeddings_batch(sequences):
    # get sequence embeddings from final layer for a batch
    tokens = tokenize_batch(sequences)
    with torch.no_grad():
        output = model(tokens, repr_layers=[last_layer], return_contacts=False)
    full_embedding = output["representations"][last_layer]  # (B, L_tokens, d)
    pad_idx = alphabet.padding_idx

    # mean embedding, average over real tokens only (ignore padding)
    mask = (tokens != pad_idx)[:, 1:-1]                # (B, L-2), True for real tokens
    reps_r = full_embedding[:, 1:-1, :]                # (B, L-2, d)

    denom = mask.sum(dim=1).clamp(min=1).unsqueeze(-1) # (B,1)
    mean_embedding = (reps_r * mask.unsqueeze(-1)).sum(dim=1) / denom

    # cls embedding, first token in sequence
    cls_embedding = full_embedding[:, 0, :]  # (B, d)
    return (
        mean_embedding.cpu().numpy(),
        cls_embedding.cpu().numpy()
    )

# load classifier and inference function

In [204]:
# shallow fully connected network
class CarbapenemaseMLP(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(emb_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # returns logits

In [205]:
def score_protein(embedding_vector: np.ndarray) -> dict:
    """
    Score a protein sequence embedding using the trained carbapenemase classifier.
    Args:
        embedding_vector: 1D numpy array of shape (emb_dim,) - the protein embedding     
    Returns:
        dict with probability, predicted class, and logit
    """
    # Load scaler params
    with open("scaler_params_fcn.json", "r") as f:
        scaler_params = json.load(f)

    scaler = StandardScaler()
    scaler.mean_ = np.array(scaler_params["mean"])
    scaler.scale_ = np.array(scaler_params["scale"])

    # load model
    emb_dim = len(scaler.mean_)
    model = CarbapenemaseMLP(emb_dim=emb_dim)
    model.load_state_dict(torch.load("final_carbapenemase_mlp_fcn.pth", map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    # scale emb
    scaled = scaler.transform(embedding_vector.reshape(1, -1))
    tensor = torch.tensor(scaled, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        logit = model(tensor)
        probability = torch.sigmoid(logit).item()

    predicted_class = "Carbapenemase" if probability >= 0.5 else "Non-carbapenemase"

    return {
        "probability": probability,
        "predicted_class": predicted_class,
        "logit": logit.item()
    }

In [206]:
embs = pd.read_csv("embeddings.csv")
kpc_embs = embs[embs["Family"] == "KPC"]
kpc_test = kpc_embs[kpc_embs["Gene"] == "KPC-11"]

In [221]:
# test seq
#kpc4 = pd.read_csv("kpc4.csv")
emb = np.fromstring(kpc_test["MeanEmbedding"].iloc[0].strip("[]"), sep=" ")

result = score_protein(emb)
#print(f"Gene name: {kpc4['Gene'].iloc[0]}")
print(f"Carbapenemase probability : {result['probability']}")
print(f"Logit                     : {result['logit']}")
print(f"Predicted class           : {result['predicted_class']}")
print(f"True label                : {kpc_test['Carbapenemase'].iloc[0]}")

#out = df[df["Gene"] == "KPC-4"]
#out.to_csv("kpc4.csv", index=False)

Carbapenemase probability : 0.8587396740913391
Logit                     : 1.8048615455627441
Predicted class           : Carbapenemase
True label                : 1


/var/folders/bz/t2rlkxxj1p791fxqscm96q_40000gn/T/ipykernel_39072/4198845954.py:3: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  emb = np.fromstring(kpc_test["MeanEmbedding"].iloc[0].strip("[]"), sep=" ")


In [208]:
kpc_test.to_csv("kpc11.csv", index=False)

# Create KPC-4 mutants:
### mutate every position (all 20 AA), extract mutant embs

In [209]:
aa_vocab = [c for c in list("ACDEFGHIKLMNPQRSTVWY")]
kpc11 = pd.read_csv("kpc11.csv")
seq = kpc11["Sequence"].iloc[0]
seq

'MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGVYAMDTGSGATVSYRAEERFPLCSSFKGFLAAAVLARSQQQAGLLDTPIRYGKNALVLWSPISEKYLTTGMTVAELSAAAVQYSDNAAANLLLKELGGPAGLTAFMRSIGDTTFRLDRWELELNSAIPGDARDTSSPRAVTESLQKLTLGSALAAPQRQQFVDWLKGNTTGNHRIRAAVPADWAVGDKTGTCGVYGTANDYAVVWPTGRAPIVLAVYTRAPNKDDKHSEAVIAAAARLALEGLGVNGQ'

In [210]:
# create mutant seqs
seq = [c for c in list(seq)]
#print(seq)

mutants = []
position = []
canonical = []
mutation = []

for i in range(0, len(seq)):
    for r in aa_vocab:
        can = kpc11["Sequence"].iloc[0][i]
        seq[i] = r
        mutant = ""
        for p in seq:
            mutant += p
        #print(mutant)
        mutants.append(mutant)
        position.append(i)
        mutation.append(r)
        canonical.append(can)
    seq = kpc11["Sequence"].iloc[0]
    seq = [c for c in list(seq)]
    #print(seq)
len(mutants)


5860

In [211]:
mutant_df = pd.DataFrame()
mutant_df["Sequence"] = mutants
mutant_df["Mutated_position"] = position
mutant_df["Canonical_residue"] = canonical
mutant_df["Mutated_residue"] = mutation
mutant_df

,Sequence,Mutated_position,Canonical_residue,Mutated_residue
0,ASLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,A
1,CSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,C
2,DSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,D
3,ESLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,E
4,FSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,F
...,...,...,...,...
5855,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,292,Q,S
5856,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,292,Q,T
5857,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,292,Q,V
5858,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,292,Q,W


In [212]:
# remove canonical seqs 
mutant_df = mutant_df.drop_duplicates(subset="Sequence")
mutant_df = mutant_df.sort_index()
mutant_df.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue
0,ASLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,A
1,CSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,C
2,DSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,D
3,ESLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,E
4,FSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,F


In [213]:
#run emb extraction for all sequences and save to new dataframe, allow 10-20 minutes for inference

emb_df = mutant_df.copy()
#test_df = emb_df.sample(n=20)

for _, row in emb_df.iterrows():
    mutant = row["Sequence"]
    mean_emb, cls_emb = extract_embeddings_batch([mutant])
    emb_df.at[_, "MeanEmbedding"] = json.dumps(mean_emb.tolist())
    emb_df.at[_, "CLSEmbedding"] = json.dumps(cls_emb.tolist())
emb_df["EmbeddingDim"] = len(mean_emb[0])

#save to csv
emb_df.to_csv("mutant_kpc4_embeddings.csv", index=False)
emb_df.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim
0,ASLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,A,"[[0.05247783660888672, -0.06884253770112991, -...","[[0.07693725824356079, -0.027117781341075897, ...",1280
1,CSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,C,"[[0.052396539598703384, -0.0690683126449585, -...","[[0.08123781532049179, -0.03102729469537735, 0...",1280
2,DSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,D,"[[0.05352051183581352, -0.06818139553070068, -...","[[0.0808483436703682, -0.03243960812687874, 0....",1280
3,ESLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,E,"[[0.05330732464790344, -0.06758446991443634, -...","[[0.08159898966550827, -0.03398491069674492, 0...",1280
4,FSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,F,"[[0.05214737355709076, -0.06939972937107086, -...","[[0.07708612829446793, -0.029716528952121735, ...",1280


In [214]:
# predict probs for mutants
prob_df = emb_df.copy()
for _, row in prob_df.iterrows():
    mean_emb = np.fromstring(row["MeanEmbedding"].strip("[]"), sep=" ")
    result = score_protein(mean_emb)
    prob_df.at[_, "Carbapenemase_Probability"] = result["probability"]
    prob_df.at[_, "Logit"] = result["logit"]
    prob_df.at[_, "Predicted_Class"] = result["predicted_class"]
    # add logit to df
prob_df.head()

/var/folders/bz/t2rlkxxj1p791fxqscm96q_40000gn/T/ipykernel_39072/1573067977.py:4: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  mean_emb = np.fromstring(row["MeanEmbedding"].strip("[]"), sep=" ")
/var/folders/bz/t2rlkxxj1p791fxqscm96q_40000gn/T/ipykernel_39072/1573067977.py:4: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  mean_emb = np.fromstring(row["MeanEmbedding"].strip("[]"), sep=" ")
/var/folders/bz/t2rlkxxj1p791fxqscm96q_40000gn/T/ipykernel_39072/1573067977.py:4: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  mean_emb = np.fromstring(row["MeanEmbedding"].strip("[]"), sep=" ")
/var/folders/bz/t2rlkxxj1p791fxqscm96q_40000gn/T/ipykernel_39072/1573067977.py:4: DeprecationWarning: string or file could not be read to its end due t

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class
0,ASLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,A,"[[0.05247783660888672, -0.06884253770112991, -...","[[0.07693725824356079, -0.027117781341075897, ...",1280,0.895214,2.145145,Carbapenemase
1,CSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,C,"[[0.052396539598703384, -0.0690683126449585, -...","[[0.08123781532049179, -0.03102729469537735, 0...",1280,0.898053,2.175773,Carbapenemase
2,DSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,D,"[[0.05352051183581352, -0.06818139553070068, -...","[[0.0808483436703682, -0.03243960812687874, 0....",1280,0.842401,1.676205,Carbapenemase
3,ESLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,E,"[[0.05330732464790344, -0.06758446991443634, -...","[[0.08159898966550827, -0.03398491069674492, 0...",1280,0.855926,1.781860,Carbapenemase
4,FSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,0,M,F,"[[0.05214737355709076, -0.06939972937107086, -...","[[0.07708612829446793, -0.029716528952121735, ...",1280,0.905739,2.262687,Carbapenemase


In [215]:
prob_df.sort_values(by="Logit", inplace=True, ascending=False)
prob_df["Logit"].describe()

count    5568.000000
mean        1.339490
std         0.867360
min        -1.961002
25%         0.836563
50%         1.502966
75%         2.020110
max         2.473341
Name: Logit, dtype: float64

In [216]:
# base logit: 2.4418728351593018
# add delta_logit column to df
base_logit = 1.8048615455627441
prob_df["Delta_Logit"] = prob_df["Logit"] - base_logit
prob_df.to_csv("mutant_kpc4_probabilities.csv", index=False)
prob_df["Delta_Logit"].describe()

count    5568.000000
mean       -0.465372
std         0.867360
min        -3.765863
25%        -0.968299
50%        -0.301896
75%         0.215248
max         0.668480
Name: Delta_Logit, dtype: float64

In [217]:
# add mean_probability column for each position, average over all mutants at that position
position_means = prob_df.groupby("Mutated_position")["Carbapenemase_Probability"].mean()
prob_df["Mean_Probability"] = prob_df["Mutated_position"].map(position_means)
prob_df.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability
1606,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,80,A,H,"[[0.05098888650536537, -0.06942420452833176, -...","[[0.07932128757238388, -0.03487767279148102, 0...",1280,0.922252,2.473341,Carbapenemase,0.668480,0.816921
5831,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,291,G,N,"[[0.05099264159798622, -0.06814805418252945, -...","[[0.07863552123308182, -0.03328976780176163, 0...",1280,0.922252,2.473340,Carbapenemase,0.668478,0.855638
3235,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,161,D,S,"[[0.050996605306863785, -0.06757387518882751, ...","[[0.07791221886873245, -0.03557412326335907, 0...",1280,0.922251,2.473332,Carbapenemase,0.668471,0.878708
3048,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,152,S,K,"[[0.0509977787733078, -0.06841420382261276, -0...","[[0.08026132732629776, -0.03629283979535103, 0...",1280,0.922251,2.473329,Carbapenemase,0.668467,0.850800
2054,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,102,L,R,"[[0.05098123103380203, -0.06845210492610931, -...","[[0.0794958844780922, -0.03500961512327194, 0....",1280,0.922251,2.473328,Carbapenemase,0.668466,0.878456


In [218]:
# add mean Logit column for each position, average over all mutants at that position
position_logit_means = prob_df.groupby("Mutated_position")["Logit"].mean()
prob_df["Mean_Logit"] = prob_df["Mutated_position"].map(position_logit_means)
prob_df.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability,Mean_Logit
1606,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,80,A,H,"[[0.05098888650536537, -0.06942420452833176, -...","[[0.07932128757238388, -0.03487767279148102, 0...",1280,0.922252,2.473341,Carbapenemase,0.668480,0.816921,1.620240
5831,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,291,G,N,"[[0.05099264159798622, -0.06814805418252945, -...","[[0.07863552123308182, -0.03328976780176163, 0...",1280,0.922252,2.473340,Carbapenemase,0.668478,0.855638,1.884432
3235,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,161,D,S,"[[0.050996605306863785, -0.06757387518882751, ...","[[0.07791221886873245, -0.03557412326335907, 0...",1280,0.922251,2.473332,Carbapenemase,0.668471,0.878708,2.046994
3048,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,152,S,K,"[[0.0509977787733078, -0.06841420382261276, -0...","[[0.08026132732629776, -0.03629283979535103, 0...",1280,0.922251,2.473329,Carbapenemase,0.668467,0.850800,1.797065
2054,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,102,L,R,"[[0.05098123103380203, -0.06845210492610931, -...","[[0.0794958844780922, -0.03500961512327194, 0....",1280,0.922251,2.473328,Carbapenemase,0.668466,0.878456,2.027360


In [219]:
# add mean Delta_Logit column for each position, average over all mutants at that position
position_delta_logit_means = prob_df.groupby("Mutated_position")["Delta_Logit"].mean()
prob_df["Mean_Delta_Logit"] = prob_df["Mutated_position"].map(position_delta_logit_means)
prob_df["EmbeedingType"] = "Mean"
prob_df.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability,Mean_Logit,Mean_Delta_Logit,EmbeedingType
1606,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,80,A,H,"[[0.05098888650536537, -0.06942420452833176, -...","[[0.07932128757238388, -0.03487767279148102, 0...",1280,0.922252,2.473341,Carbapenemase,0.668480,0.816921,1.620240,-0.184622,Mean
5831,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,291,G,N,"[[0.05099264159798622, -0.06814805418252945, -...","[[0.07863552123308182, -0.03328976780176163, 0...",1280,0.922252,2.473340,Carbapenemase,0.668478,0.855638,1.884432,0.079570,Mean
3235,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,161,D,S,"[[0.050996605306863785, -0.06757387518882751, ...","[[0.07791221886873245, -0.03557412326335907, 0...",1280,0.922251,2.473332,Carbapenemase,0.668471,0.878708,2.046994,0.242132,Mean
3048,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,152,S,K,"[[0.0509977787733078, -0.06841420382261276, -0...","[[0.08026132732629776, -0.03629283979535103, 0...",1280,0.922251,2.473329,Carbapenemase,0.668467,0.850800,1.797065,-0.007797,Mean
2054,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,102,L,R,"[[0.05098123103380203, -0.06845210492610931, -...","[[0.0794958844780922, -0.03500961512327194, 0....",1280,0.922251,2.473328,Carbapenemase,0.668466,0.878456,2.027360,0.222499,Mean


In [222]:
# add Delta Probability column to df
base_probability = 0.8587396740913391
prob_df["Delta_Probability"] = prob_df["Carbapenemase_Probability"] - 0.8587396740913391
prob_df.head()


,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability,Mean_Logit,Mean_Delta_Logit,EmbeedingType,Delta_Probability
1606,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,80,A,H,"[[0.05098888650536537, -0.06942420452833176, -...","[[0.07932128757238388, -0.03487767279148102, 0...",1280,0.922252,2.473341,Carbapenemase,0.668480,0.816921,1.620240,-0.184622,Mean,0.063512
5831,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,291,G,N,"[[0.05099264159798622, -0.06814805418252945, -...","[[0.07863552123308182, -0.03328976780176163, 0...",1280,0.922252,2.473340,Carbapenemase,0.668478,0.855638,1.884432,0.079570,Mean,0.063512
3235,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,161,D,S,"[[0.050996605306863785, -0.06757387518882751, ...","[[0.07791221886873245, -0.03557412326335907, 0...",1280,0.922251,2.473332,Carbapenemase,0.668471,0.878708,2.046994,0.242132,Mean,0.063511
3048,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,152,S,K,"[[0.0509977787733078, -0.06841420382261276, -0...","[[0.08026132732629776, -0.03629283979535103, 0...",1280,0.922251,2.473329,Carbapenemase,0.668467,0.850800,1.797065,-0.007797,Mean,0.063511
2054,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,102,L,R,"[[0.05098123103380203, -0.06845210492610931, -...","[[0.0794958844780922, -0.03500961512327194, 0....",1280,0.922251,2.473328,Carbapenemase,0.668466,0.878456,2.027360,0.222499,Mean,0.063511


In [223]:
# save df
prob_df.to_csv("mutant_kpc11_probabilities.csv", index=False)

In [225]:
prob_df.columns

Index(['Sequence', 'Mutated_position', 'Canonical_residue', 'Mutated_residue',
       'MeanEmbedding', 'CLSEmbedding', 'EmbeddingDim',
       'Carbapenemase_Probability', 'Logit', 'Predicted_Class', 'Delta_Logit',
       'Mean_Probability', 'Mean_Logit', 'Mean_Delta_Logit', 'EmbeedingType',
       'Delta_Probability'],
      dtype='object')

In [233]:
# add delta logit mean for that position, average over all mutants at that position
prob_df["Mean_Delta_Logit"] = prob_df.groupby("Mutated_position")["Delta_Logit"].transform("mean")

# add delta probability for that position, average over all mutants at that position
prob_df["Mean_Delta_Probability"] = prob_df.groupby("Mutated_position")["Delta_Probability"].transform("mean")
prob_df.head()
prob_df.to_csv("mutant_kpc11_probabilities.csv", index=False)

In [239]:
# sort df by Mean_Delta_Logit
#group df by mutated position and get mean delta logit for each position, then sort by that
grouped_prob_df = prob_df.groupby("Mutated_position")["Mean_Delta_Logit"].mean().sort_values(ascending=True)
grouped_prob_df.head(10)

Mutated_position
255   -2.544435
256   -2.383721
271   -2.299009
257   -2.289351
43    -2.262531
5     -2.150993
105   -2.122480
249   -1.961911
114   -1.929092
12    -1.740348
Name: Mean_Delta_Logit, dtype: float64

In [241]:
# compute min logit per position and add to prob_df
min_logit_per_position = prob_df.groupby("Mutated_position")["Logit"].min()
prob_df["Min_Logit"] = prob_df["Mutated_position"].map(min_logit_per_position)
prob_df.head()

,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability,Mean_Logit,Mean_Delta_Logit,EmbeedingType,Delta_Probability,Mean_Delta_Probability,Min_Logit
5103,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,E,"[[0.05796727538108826, -0.06942269206047058, -...","[[0.08032112568616867, -0.03459315374493599, 0...",1280,0.255149,-1.071336,Non-carbapenemase,-2.876197,0.341683,-0.739574,-2.544435,Mean,-0.603590,-0.517057,-1.886602
5109,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,L,"[[0.06107009947299957, -0.06913203001022339, -...","[[0.07972662150859833, -0.034166812896728516, ...",1280,0.131632,-1.886602,Non-carbapenemase,-3.691463,0.341683,-0.739574,-2.544435,Mean,-0.727107,-0.517057,-1.886602
5108,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,K,"[[0.05409065634012222, -0.06780257821083069, -...","[[0.08035734295845032, -0.03513117879629135, 0...",1280,0.797964,1.373620,Carbapenemase,-0.431241,0.341683,-0.739574,-2.544435,Mean,-0.060775,-0.517057,-1.886602
5113,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,Q,"[[0.057415228337049484, -0.06862905621528625, ...","[[0.0809495598077774, -0.03441504016518593, 0....",1280,0.320343,-0.752197,Non-carbapenemase,-2.557058,0.341683,-0.739574,-2.544435,Mean,-0.538397,-0.517057,-1.886602
5117,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,V,"[[0.059363480657339096, -0.0705282986164093, -...","[[0.07907991856336594, -0.03507959470152855, 0...",1280,0.163617,-1.631557,Non-carbapenemase,-3.436419,0.341683,-0.739574,-2.544435,Mean,-0.695122,-0.517057,-1.886602


In [243]:
# rank 20 positions by min logit
ranked_positions = prob_df.groupby("Mutated_position")["Min_Logit"].min().sort_values(ascending=True).head(20)
ranked_positions


Mutated_position
249   -1.961002
256   -1.909705
255   -1.886602
261   -1.757704
114   -1.651530
7     -1.577451
257   -1.568980
70    -1.545975
87    -1.501498
271   -1.474577
98    -1.466300
184   -1.438717
12    -1.433088
105   -1.419555
43    -1.369350
32    -1.351197
194   -1.350125
202   -1.334370
21    -1.296130
92    -1.273566
Name: Min_Logit, dtype: float64

In [246]:
# add Max Logit per position to the prob_df
max_logit_per_position = prob_df.groupby("Mutated_position")["Logit"].max()
prob_df["Max_Logit"] = prob_df["Mutated_position"].map(max_logit_per_position)
prob_df.head()


,Sequence,Mutated_position,Canonical_residue,Mutated_residue,MeanEmbedding,CLSEmbedding,EmbeddingDim,Carbapenemase_Probability,Logit,Predicted_Class,Delta_Logit,Mean_Probability,Mean_Logit,Mean_Delta_Logit,EmbeedingType,Delta_Probability,Mean_Delta_Probability,Min_Logit,Max_Logit
5103,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,E,"[[0.05796727538108826, -0.06942269206047058, -...","[[0.08032112568616867, -0.03459315374493599, 0...",1280,0.255149,-1.071336,Non-carbapenemase,-2.876197,0.341683,-0.739574,-2.544435,Mean,-0.603590,-0.517057,-1.886602,1.37362
5109,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,L,"[[0.06107009947299957, -0.06913203001022339, -...","[[0.07972662150859833, -0.034166812896728516, ...",1280,0.131632,-1.886602,Non-carbapenemase,-3.691463,0.341683,-0.739574,-2.544435,Mean,-0.727107,-0.517057,-1.886602,1.37362
5108,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,K,"[[0.05409065634012222, -0.06780257821083069, -...","[[0.08035734295845032, -0.03513117879629135, 0...",1280,0.797964,1.373620,Carbapenemase,-0.431241,0.341683,-0.739574,-2.544435,Mean,-0.060775,-0.517057,-1.886602,1.37362
5113,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,Q,"[[0.057415228337049484, -0.06862905621528625, ...","[[0.0809495598077774, -0.03441504016518593, 0....",1280,0.320343,-0.752197,Non-carbapenemase,-2.557058,0.341683,-0.739574,-2.544435,Mean,-0.538397,-0.517057,-1.886602,1.37362
5117,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,255,P,V,"[[0.059363480657339096, -0.0705282986164093, -...","[[0.07907991856336594, -0.03507959470152855, 0...",1280,0.163617,-1.631557,Non-carbapenemase,-3.436419,0.341683,-0.739574,-2.544435,Mean,-0.695122,-0.517057,-1.886602,1.37362


In [249]:
# rank top 20 positions by max logit
ranked_positions_max = prob_df.groupby("Mutated_position")["Max_Logit"].max().sort_values(ascending=False).head(20)
ranked_positions_max

Mutated_position
80     2.473341
291    2.473340
161    2.473332
152    2.473329
96     2.473328
102    2.473328
163    2.473324
17     2.473320
268    2.473318
82     2.473309
174    2.473303
24     2.473294
103    2.473292
108    2.473289
171    2.473257
193    2.473220
221    2.473214
166    2.473213
3      2.473166
126    2.473160
Name: Max_Logit, dtype: float64

In [250]:
# save prob_df with all new columns
prob_df.to_csv("mutant_kpc11_probabilities.csv", index=False)


In [220]:
# ESM FOLD for high prob drop mutants compared to canonical 
# differences between mutant and canonical seqs with large drops in predicted carbapenemase probability may indicate important residues for function, which can be further investigated with ESM FOLD to predict structural impacts of mutations.